# Template for Multi-agent Personal Assistant System Built on Supervisor Pattern

_Building a simple multi-agent personal assistant system following a supervisor/manager and sub-agents pattern._

Build a supervisor agent that should coordinates the following two specialized workers or sub-agents. Once each sub-agent completes its task, the supervisor should synthesize both results into a coherent response to share with the user.

- A calender agent that handles availability check, schedulning and event management.
- An email agent that drafts messages, manages communication and send notifications.

In [ ]:
# Import packages

# Import class `ChatOllama` from module `langchain_ollama`
# Import function `tool` from module `langchain.tools`
# Import function `create_agent` from module `langchain.agents`
# Import module class `datetime` from module `datetime`

## Model

_Connecting an appropriate model to a chat client._

In [3]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "qwen3.5:2b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [ ]:
# Initialize a chat client by calling constuctor of class ChatOllama with the following arguments.
# 1. model name against parameter `model`, 
# 2. 'False` against parameter `reasoning`, and
# 3. model endpoint to parameter `base_url`, and
# store the reference to the created instance in a variable called `model`.

# Code here
# ...

## Tools

Defines all the tools by the system.

In [5]:
@tool
def create_calender_event(
    title: str,
    start_time: str,        # ISO format: "2024-01-15T14:00:00"
    end_time: str,          # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],   # Email addresses
    location: str,    
) -> str:
    """Creates a calender event. Requires datetime in ISO format."""

    # A dummy function for this experiment. In practice, this would call Google Calendar API, Outlook API, etc.

    return f"Event was created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

In [6]:
@tool
def send_email(
    to: list[str],          # Email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""

    # A dummy function for this experiment. In practice, this would call SendGrid, Gmail API, etc.

    return f"Email sent to {', '.join(to)} - Subject: {subject}"

In [7]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,              # ISO format: "2024-01-15"
    duration_minutes: int,
) -> list[str]:
    """Check calendar availability for given attendees on a specific date. Requires date in ISO format."""

    # A dummy function for this experiment. In practice, this would query calendar APIs

    return ["09:00", "14:00", "16:00"]

## Sub-Agents
_Creating specialized sub-agents._

### Calendar Agent
The calendar agent understands natural language scheduling requests and translates them into precise API calls. It handles date parsing, availability checking, and event creation.

In [ ]:
CALENDAR_AGENT_PROMPT = f"""
    You are a calendar scheduling assistant.
    Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm')
    into proper ISO datetime formats. Today is {datetime.date(datetime.now()).strftime("%A")}.
    Current time is {datetime.now().isoformat()} (in ISO format).
    Use 'get_available_time_slots' to check availability when needed.
    If there is no suitable time slot, stop and confirm unavailability in your response.
    Use 'create_calendar_event' to schedule events.
    Always confirm what was scheduled in your final response.
"""

# Create the calendar agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing tool function `create_calender_event` and `get_available_time_slots` against parameter `tools`, and
# 3. the calender agent prompt against parameter `system_prompt`, and
# store the reference of the created instance into a variable called `calendar_agent`.

# Code here
# ...

In [ ]:

query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

# Test the calendar agent to check how it handles event scheduling in natural language
# by invoking the agent with the following arguments
# 1. Passing `{"messages": [{"role": "user", "content": query}]}` against the first (positioned) parameter,
# 2. "values" against parameter `stream_mode`, and
# 3. "v2" against parameter `version`, and
# store the output into a variable `agent_output`.
# NOTE THAT THE STEP TAKE NEARLY A MINUTE TO COMPLETE

# Code here
# ...


# Now, do the analysis by printing the all the messages received from the agent
# by iterating over `agent_output.value["messages"]` in a loop and printing each message
# calling its function `pretty_print` without any argument.

# Code here
# ...

### Email Agent
The email agent handles message composition and sending. It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [ ]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use 'send_email' to send the message. "
    "Always confirm what was sent in your final response."
)

# Create the email agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing only tool function `send_email` against parameter `tools`, and
# 3. the email agent prompt against parameter `system_prompt`, and
# store the reference of the created instance into a variable called `email_agent`.

# Code here
# ...

In [ ]:
query = "Send the quality assurance team a reminder about reviewing the new test cases."

# Test the email agent to check how it handles email related instructions in natural language
# by invoking the agent with the following arguments
# 1. Passing `{"messages": [{"role": "user", "content": query}]}` against the first (positioned) parameter,
# 2. "values" against parameter `stream_mode`, and
# 3. "v2" against parameter `version`, and
# store the output into a variable `agent_output`.
# NOTE THAT THE STEP TAKE NEARLY A MINUTE TO COMPLETE

# Code here
# ...

# Now, do the analysis by printing the all the messages received from the agent
# by iterating over `agent_output.value["messages"]` in a loop and printing each message
# calling its function `pretty_print` without any argument.

# Code here
# ...

## Sub-agents as Tools

Wrapping each sub-agent as a tool that the supervisor can invoke creating the layered system in which each sub-agent has a narrow focus with domain-specific tools and prompts. The supervisor will see high-level tools like “schedule_event”, not low-level tools like “create_calendar_event”.

In [12]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """

    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})

    return result["messages"][-1].text      # Returns the result extracted from last message

In [13]:
@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about the meeting')
    """
    
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })

    return result["messages"][-1].text      # Returns the result extracted from last message

## Supervisor Agent
Creates the supervisor that orchestrates the sub-agents. The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual tools/API level.

In [ ]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

# Create the supervisor agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing tool function `schedule_event` and `manage_email` against parameter `tools`, and
# 3. the supervisor prompt against parameter `system_prompt`, and
# store the reference of the created instance into a variable called `supervisor_agent`.

# Code here
# ...

## Testing
 Tests complete system with complex requests that require coordination across multiple domains.

### Testing over simple single-domain (calendar) request
The supervisor is expected to identify this as a calendar task, call `schedule_event`, and the calendar agent should handle date parsing and event creation.

In [ ]:
# MAY TAKE AROUND 1 AND ½ MINUTES TO COMPLETE

query = "Schedule a team standup for tomorrow at 9am"

# Invoke the supervisor agent with the following arguments
# 1. Passing `{"messages": [{"role": "user", "content": query}]}` against the first (positioned) parameter,
# 2. "values" against parameter `stream_mode`, and
# 3. "v2" against parameter `version`, and
# store the output into a variable `agent_output`.
# NOTE THAT THE STEP TAKE NEARLY TWO MINUTES TO COMPLETE

# Code here
# ...

# Now, do the analysis by printing the all the messages received from the agent
# by iterating over `agent_output.value["messages"]` in a loop and printing each message
# calling its function `pretty_print` without any argument.

# Code here
# ...

### Testing over multiple-domain (calender + email) request
The supervisor should recognize that this requires both calendar and email actions, call `schedule_event` for the meeting, then call `manage_email` for the reminder. After each sub-agent completes its task, and the supervisor should synthesize both results into a coherent response.

In [ ]:
# MAY TAKE AROUND 3 AND ½ MINUTES TO COMPLETE

query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

# Invoke the supervisor agent with the following arguments
# 1. Passing `{"messages": [{"role": "user", "content": query}]}` against the first (positioned) parameter,
# 2. "values" against parameter `stream_mode`, and
# 3. "v2" against parameter `version`, and
# store the output into a variable `agent_output`.
# NOTE THAT THE STEP TAKE NEARLY FOUR MINUTES TO COMPLETE

# Code here
# ...

# Now, do the analysis by printing the all the messages received from the agent
# by iterating over `agent_output.value["messages"]` in a loop and printing each message
# calling its function `pretty_print` without any argument.

# Code here
# ...